## Linux `crontab` Command
Used to schedule recurring tasks in Linux. The `crontab` command allows you to create, edit, and manage cron jobs, which are scripts or commands that run automatically at specified intervals.

Check the current crontab entries for the `dennie` user:

In [ ]:
dennie@HP:~/linux-tut$ crontab -l
no crontab for dennie

There are currently no cron jobs scheduled for the `dennie` user.

Next, let's create a new cron job that runs a simple script every minute. Create a new file called `hello.sh` in the `labex/project` directory with the following content:

In [ ]:
dennie@HP:~/linux-tut$ cat > labex/project/hello.sh << 'EOF'
> #!/bin/bash
echo "Hello from cron job!"
> EOF

In [ ]:
dennie@HP:~/linux-tut$ cat labex/project/hello.sh

In [ ]:
#!/bin/bash
echo "Hello from cron job!"

Make the script executable:

In [ ]:
dennie@HP:~/linux-tut$ chmod +x labex/project/hello.sh

Now, let's add a new cron job to run this script every minute:

In [ ]:
dennie@HP:~/linux-tut$ crontab -e

In [ ]:
GNU nano 7.2         /tmp/crontab.oV6VMU/crontab                                                                                                  
# Edit this file to introduce tasks to be run by cron.
# 
# Each task to run has to be defined through a single line
# indicating with different fields when the task will be run
# and what command to run for the task
#

This will open the crontab editor. Add the following line at the end of the file:

In [ ]:
GNU nano 7.2         /tmp/crontab.oV6VMU/crontab                                                                                                  
# Edit this file to introduce tasks to be run by cron.
# 
# Each task to run has to be defined through a single line
# indicating with different fields when the task will be run
# and what command to run for the task
* * * * * /home/dennie/linux-tut/labex/project/hello.sh

In [ ]:
dennie@HP:~/linux-tut$ crontab -e
no crontab for dennie - using an empty one
crontab: installing new crontab

This cron job will run the `hello.sh` script every minute.

Save and exit the crontab editor.

To verify that the cron job is running, wait for a minute and check the system log:

In [ ]:
dennie@HP:~/linux-tut$ tail -n 5 /var/log/syslog

In [ ]:
2026-09-13T15:11:17.845163+00:00 HP systemd-resolved[120]: Clock change detected. Flushing caches.
2026-09-13T15:11:49.329795+00:00 HP systemd-resolved[120]: Clock change detected. Flushing caches.
2026-09-13T15:12:00.421826+00:00 HP CRON[18175]: (dennie) CMD (/home/dennie/linux-tut/labex/project/hello.sh )
2026-09-13T15:12:00.436885+00:00 HP CRON[18174]: (CRON) info (No MTA installed, discarding output)
2026-09-13T15:12:20.883538+00:00 HP systemd-resolved[120]: Clock change detected. Flushing caches.

Found it — your notebook confirms the cron job **did** fire, but syslog is only showing you that cron _executed_ the command, not what the command _printed_.

**What's happening:** Look at the second log line right after your job ran:

In [ ]:
2026-09-13T15:12:00.436885+00:00 HP CRON[18174]: (CRON) info (No MTA installed, discarding output)

**DIAGNOSIS PATH:**

1. cron's syslog entry (`CRON[18175]: (dennie) CMD (...)`) only proves the scheduler invoked the script — it says nothing about the script's stdout/stderr.
2. By default, cron captures a job's stdout/stderr and tries to **email** it to the user via the local MTA (mail transfer agent — think `postfix`/`sendmail`).
3. Your WSL box has no MTA installed, so cron logs "discarding output" and your `echo "Hello from cron job!"` goes straight into the void. It never touches syslog at all.

**Developer bridge:** this is the same trap as running `python manage.py mycommand &` or a Celery beat task and expecting `print()` statements to show up in a log file you never pointed it at — output goes nowhere unless you redirect it explicitly.

**TRY NOW** — fix it by redirecting the script's output to a file instead of relying on mail:

In [ ]:
dennie@HP:~/linux-tut$ crontab -e

In [ ]:
* * * * * /home/dennie/linux-tut/labex/project/hello.sh >> /home/dennie/linux-tut/labex/project/hello.log 2>&1

Save and exit.

In [ ]:
dennie@HP:~/linux-tut$ crontab -e
crontab: installing new crontab

Then after a minute:

In [ ]:
dennie@HP:~/linux-tut$ tail -f /home/dennie/linux-tut/labex/project/hello.log
Hello from cron job!
Hello from cron job!


**READ THE OUTPUT:** you should now see `Hello from cron job!` appended once per minute in that file — that's your script's actual output, separate from cron's own activity log in syslog.

**GOTCHA (WSL-specific too):** "No MTA installed" is common on minimal Ubuntu/WSL installs — real production Linux boxes often _do_ have `postfix` or `exim` running, so this exact failure mode is WSL/lightweight-install-specific. In a real support ticket, if a coworker says "my cron job's output isn't showing up anywhere," checking for `MTA` and checking whether they redirected output is step one.

### Schedule a Recurring Task with Crontab
Schedule a recurring task using the crontab command.

Create a new script called `backup.sh` in the `labex/project` directory that will perform a simple backup operation:

In [ ]:
dennie@HP:~/linux-tut$ cat > labex/project/backup.sh << 'EOF'
#!/bin/bash
echo "Backing up data to backup.txt"
date >> /home/dennie/linux-tut/labex/project/backup.txt
EOF

In [ ]:
dennie@HP:~/linux-tut$ cat labex/project/backup.sh 

In [ ]:
#!/bin/bash
echo "Backing up data to backup.txt"
date >> /home/dennie/linux-tut/labex/project/backup.txt

Make the script executable:

In [ ]:
dennie@HP:~/linux-tut$ chmod +x labex/project/backup.sh

Now, let's schedule this script to run every 3 minutes using crontab:

In [ ]:
dennie@HP:~/linux-tut$ crontab -e

In [ ]:
*/3 * * * * /home/dennie/linux-tut/labex/project/backup.sh >> /home/dennie/linux-tut/labex/project/backup.txt 2>&1

In [ ]:
dennie@HP:~/linux-tut$ crontab -e
no crontab for dennie - using an empty one
crontab: installing new crontab

### Manage Crontab Entries
Manage your crontab entries, including listing, editing, and removing cron jobs.

List all the current crontab entries for the `dennie` user:

In [ ]:
crontab -l

You should see the two cron jobs you created in the previous steps:

In [ ]:
* * * * * /home/labex/project/hello.sh
*/5 * * * * /home/labex/project/backup.sh

To edit the crontab, use the following command:

In [ ]:
crontab -e

This will open the crontab editor, where you can add, modify, or remove cron job entries.

Let's try to remove the `hello.sh` cron job. In the crontab editor, locate the line `* * * * * /home/labex/project/hello.sh` and delete it. Save and exit the editor.

To verify that the `hello.sh` cron job has been removed, run the following command again:

In [ ]:
crontab -l

You should only see the `backup.sh` cron job remaining.

Now, let's disable the `backup.sh` cron job by adding a `#` at the beginning of the line:

In [ ]:
crontab -e

Add a `#` at the beginning of the line `*/5 * * * * /home/labex/project/backup.sh`. Save and exit the editor.

To verify that the `backup.sh` cron job has been disabled, run the following command:

In [ ]:
crontab -l

You should see the line for `backup.sh` with a `#` at the beginning, indicating that it is now disabled.